# Positive semi-definite matrices

_Linear Algebra_

**Symmetric matrices that never bend the quadratic the wrong way: a bowl, not a saddle.**

A symmetric matrix $A$ is positive semi-definite (PSD) if the quadratic form $x^\top A x$ is never negative, for any $x$.

     Geometrically, the surface $x^\top A x$ is an upward bowl: it curves up in every direction (or is flat), never down. A matrix that curves down somewhere gives a saddle and is not PSD.

---

This notebook builds the idea up one step at a time: first construct a matrix that *must* be PSD, then verify it two independent ways, then see how its eigenvalues differ from an indefinite matrix. Run each cell and read the note above it. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — NumPy

We verify the PSD property in three stages: (1) build a matrix guaranteed to be PSD, (2) check it via its eigenvalues, (3) spot-check the defining inequality $x^\top M x \ge 0$ directly on random vectors.

### Step 1 — Build a matrix that must be PSD

For *any* matrix $A$, the product $A^\top A$ is always symmetric and positive semi-definite. This is the easiest way to manufacture a PSD matrix on demand. We draw a random $4\times3$ matrix and form $M = A^\top A$, then confirm it equals its own transpose (symmetry is required before PSD even makes sense).

In [ ]:
rng = np.random.default_rng(0)

# For any A, the product A^T A is symmetric and positive semi-definite.
A = rng.standard_normal((4, 3))
M = A.T @ A

# Symmetry check: M should equal its own transpose.
is_symmetric = np.allclose(M, M.T)
print("symmetric:", is_symmetric)

### Step 2 — Confirm PSD via the eigenvalues

A symmetric matrix is PSD exactly when all of its eigenvalues are non-negative. We use `np.linalg.eigvalsh`, the solver specialized for symmetric (Hermitian) matrices, which returns real eigenvalues. A tiny negative tolerance (`-1e-9`) absorbs floating-point round-off so values that are mathematically zero aren't flagged as negative.

In [ ]:
# PSD <=> every eigenvalue >= 0. eigvalsh is the symmetric-matrix solver.
eigenvalues = np.linalg.eigvalsh(M)
print("eigenvalues:", np.round(eigenvalues, 4))

# Allow a tiny tolerance for floating-point round-off.
all_nonnegative = np.all(eigenvalues >= -1e-9)
print("all >= 0 (PSD):", all_nonnegative)

### Step 3 — Spot-check the quadratic form directly

The definition of PSD is that $x^\top M x \ge 0$ for *every* vector $x$. We can't test infinitely many vectors, but we can sample many random ones and confirm the minimum value we see is still non-negative. The `einsum` call computes $x^\top M x$ for all 1000 sampled vectors at once.

In [ ]:
# Sample many random vectors x and evaluate x^T M x for each.
xs = rng.standard_normal((1000, 3))
quad = np.einsum("ni,ij,nj->n", xs, M, xs)

# If M is PSD, even the smallest value over all samples should be >= 0.
smallest = quad.min()
print("min x^T M x over samples:", round(smallest, 6), ">= 0")

## Visualize the data & results

_How do the eigenvalues of a PSD matrix differ from those of an indefinite one?_

### Step 1 — Build one PSD and one indefinite matrix

To contrast the two cases we need a matrix of each kind. As before, $A^\top A$ is guaranteed PSD. For the opposite case we symmetrize a *square* random matrix as $B + B^\top$ — this is symmetric (so its eigenvalues are real) but generally **indefinite**, meaning some eigenvalues are positive and some negative.

In [ ]:
rng = np.random.default_rng(0)

# Symmetric PSD matrix: all eigenvalues >= 0.
A = rng.standard_normal((4, 3))
M = A.T @ A
psd = np.linalg.eigvalsh(M)

# Symmetric but indefinite matrix: some eigenvalues go negative.
B = rng.standard_normal((3, 3))
S = B + B.T
indef = np.linalg.eigvalsh(S)

### Step 2 — Plot the two eigenvalue spectra side by side

The left bars are the eigenvalues of the PSD matrix — all reach upward (non-negative). On the right, the indefinite matrix has at least one bar dipping below zero; we color negative bars red to make the sign obvious. This is the visual signature that separates a "bowl" (PSD) from a "saddle" (indefinite).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4))

# Left — eigenvalues of the PSD matrix (all non-negative).
ax1.bar(['lambda 1', 'lambda 2', 'lambda 3'], psd, color='#7ee787')
ax1.set_title('eigenvalues of A.T A (PSD)')

# Right — eigenvalues of the indefinite matrix; red where negative.
colors = ['#ff7b72' if v < 0 else '#7ee787' for v in indef]
ax2.bar(['lambda 1', 'lambda 2', 'lambda 3'], indef, color=colors)
ax2.set_title('symmetric but indefinite')
ax2.axhline(0, color='#9aa7b4', linewidth=0.8)

plt.tight_layout()
plt.show()